## Quiet the logs

`scarlets/utils/RedisLogger.py` calls `logging.basicConfig(level=DEBUG)` at
**import time**, which sets the *root* logger to DEBUG for the whole kernel -
so every library that logs (redis-py, httpx, asyncio) starts emitting DEBUG
into your cell output as soon as anything imports `scarlets`.

Run this first. It configures the root logger before `scarlets` gets the
chance, which makes that `basicConfig` call a no-op (`basicConfig` does
nothing when the root logger already has handlers).

If you ever see lines like `Failed to enable maintenance notifications:
unknown subcommand 'MAINT_NOTIFICATIONS'` - those are harmless. redis-py 8.x
probes for a Redis 8 maintenance feature on every new connection, Redis 7.x
replies "unknown subcommand", and the client logs it at DEBUG and carries on
with a perfectly normal connection.

In [1]:
import logging

# Claim the root logger first. logging.basicConfig() does nothing if the
# root logger already has handlers, so configuring it here makes
# RedisLogger's own basicConfig(level=DEBUG) - which runs when scarlets is
# imported further down - a no-op instead of a takeover.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# Belt and braces: if something already imported scarlets in this kernel
# (a re-run, or an earlier notebook), the call above was itself the no-op,
# so set the level explicitly too.
logging.getLogger().setLevel(logging.WARNING)
for _noisy in ("redis", "httpx", "httpx2", "httpcore", "httpcore2",
               "urllib3", "asyncio", "mcp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

print("log level: WARNING (set INFO or DEBUG here if you want the detail back)")

log level: WARNING (set INFO or DEBUG here if you want the detail back)


# 09 — Self-healing distributed computation

Each worker reports its **current capacity and health status** via CSV files
(`worker{number}.csv` and `worker{number}_tasks.csv`). The LLM head reads all
capacity reports, computes load balancing decisions, and redistributes work
when workers fail or become overloaded.

## Self-healing capabilities

| Capability | How it works |
|------------|--------------|
| Health monitoring | Workers report CPU, memory, active tasks |
| Failure detection | Missing or stale capacity reports indicate failure |
| Load balancing | Head redistribute tasks to healthy workers |
| Work recovery | Failed task reassignment to available workers |

## CSV data flow

Each worker maintains two CSV files:

- `worker{number}.csv`: cpu_pct, memory_mb, active_tasks, estimated_capacity, heartbeat
- `worker{number}_tasks.csv`: task_id, complexity, priority, estimated_duration, status

The head reads all capacity reports, decides workload distribution,
and writes assignment decisions to shared CSV.

## Parameters

In [2]:
import os, time, json, subprocess, random

HEAD_APP_ID   = os.environ.get("HEAD_APP_ID", "scarlet-agent-head")
WORKER_APP_ID = os.environ.get("WORKER_APP_ID", "scarlet-agent-worker")

HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1,worker2,worker3,worker4").split(",")
                if n.strip()]

HEAD_BUS      = os.environ.get("HEAD_BUS", f"{HEAD_APP_ID}_headagent")
HEAD_HOST     = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

_explicit_ids = os.environ.get("EXPECTED_AGENT_IDS", "").strip()
if _explicit_ids:
    EXPECTED_AGENT_IDS = {i.strip() for i in _explicit_ids.split(",") if i.strip()}
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = EXPECTED_AGENT_IDS - {HEAD_AGENT_ID}
else:
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = {f"{WORKER_APP_ID}_{n}" for n in WORKER_NODES}
    EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

print(f"head app id   : {HEAD_APP_ID}")
print(f"worker app id : {WORKER_APP_ID}")
print(f"workers       : {WORKER_NODES}")
print(f"expecting     : {sorted(EXPECTED_AGENT_IDS)}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

head app id   : loop-agent-head
worker app id : loop-agent-worker
workers       : ['worker1', 'worker2', 'worker3', 'worker4']
expecting     : ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']
fleet managed : Gustavo


## Generate initial state: capacity reports and task queues

Worker 3 will be simulated as degraded (high CPU) to demonstrate
self-healing behavior.

In [3]:
import pandas as pd

DATA_DIR = os.environ.get("DATA_DIR", "/data")

def write_worker_csv(path, df):
    df.to_csv(path, index=False)
    print(f"{path}: {len(df)} rows")

rng = random.Random(20260930)
os.makedirs(DATA_DIR, exist_ok=True)

for i, node in enumerate(WORKER_NODES, start=1):
    n_reports = 10
    reports = []
    
    for j in range(n_reports):
        if i == 3:
            cpu = min(90 + rng.randint(-5, 10), 100)
            memory = 7500 + rng.randint(-500, 500)
            active = 8 + rng.randint(0, 3)
        else:
            cpu = 25 + rng.randint(-10, 15)
            memory = 4000 + rng.randint(-1000, 1000)
            active = 2 + rng.randint(0, 2)
        
        capacity = max(0, 100 - cpu)
        
        reports.append({
            "timestamp": f"2026-09-30T{10 + (j*3):02d}:00:00",
            "cpu_pct": cpu,
            "memory_mb": memory,
            "active_tasks": active,
            "estimated_capacity": capacity,
            "heartbeat": "alive"
        })
    
    df = pd.DataFrame(reports)
    path = f"{DATA_DIR}/{node}.csv"
    write_worker_csv(path, df)
    
    n_tasks = 5
    tasks = []
    for j in range(n_tasks):
        tasks.append({
            "task_id": f"{node}_task_{j:03d}",
            "complexity": rng.randint(1, 100),
            "priority": rng.choice(["low", "medium", "high", "critical"]),
            "estimated_duration": rng.randint(5, 60),
            "status": "pending"
        })
    
    df = pd.DataFrame(tasks)
    path = f"{DATA_DIR}/{node}_tasks.csv"
    write_worker_csv(path, df)

# ─── Declare the CSVs as NAMED data sources ────────────────────────────────
#
# Writing the files is not enough. sum/median reach a worker's data through
# CSV_PATH with a hardcoded "SELECT value FROM data" - which cannot work here,
# because these health reports have no `value` column at all (they have
# cpu_pct, memory_mb, active_tasks, estimated_capacity, heartbeat).
#
# query_feature and list_tags CAN read arbitrary columns, and they go through
# the same CsvConnector - but they address a source by NAME, resolved from the
# worker's own ~/.scarlet/config.yaml (or SCARLET_LOCAL_CONFIG). With only
# CSV_PATH set, find_source() returns None on every worker and a query dies
# with "no worker holding data source 'x' responded within 15.0s" - which is
# exactly what this notebook did before this cell existed.
#
# So each worker gets a small config naming its two files. The compose file
# points SCARLET_LOCAL_CONFIG at the matching path in this same shared volume.
import yaml

for node in WORKER_NODES:
    sources = {
        "sources": [
            {
                "name": f"{node}_health",
                "type": "csv",
                "mode": "local",
                "description": (
                    "Worker health reports. Columns: timestamp, cpu_pct, "
                    "memory_mb, active_tasks, estimated_capacity, heartbeat."
                ),
                "path": f"{DATA_DIR}/{node}.csv",
            },
            {
                "name": f"{node}_tasks",
                "type": "csv",
                "mode": "local",
                "description": (
                    "Task assignments held by this worker. Columns: task_id, "
                    "status, estimated_minutes."
                ),
                "path": f"{DATA_DIR}/{node}_tasks.csv",
            },
        ]
    }
    # The worker reads its real ~/.scarlet/config.yaml now - HOME points
    # into this shared volume - so there is no SCARLET_LOCAL_CONFIG
    # override and no per-file naming convention to rely on.
    _scarlet_dir = os.path.join(DATA_DIR, node, ".scarlet")
    os.makedirs(_scarlet_dir, exist_ok=True)
    cfg_path = os.path.join(_scarlet_dir, "config.yaml")
    with open(cfg_path, "w") as f:
        yaml.safe_dump(sources, f, sort_keys=False)
    print(f"{node}: declared {len(sources['sources'])} named sources -> {cfg_path}")

/data/worker1.csv: 10 rows
/data/worker1_tasks.csv: 5 rows
/data/worker2.csv: 10 rows
/data/worker2_tasks.csv: 5 rows
/data/worker3.csv: 10 rows
/data/worker3_tasks.csv: 5 rows
/data/worker4.csv: 10 rows
/data/worker4_tasks.csv: 5 rows
worker1: declared 2 named sources -> /data/worker1/.scarlet/config.yaml
worker2: declared 2 named sources -> /data/worker2/.scarlet/config.yaml
worker3: declared 2 named sources -> /data/worker3/.scarlet/config.yaml
worker4: declared 2 named sources -> /data/worker4/.scarlet/config.yaml


## Simulate failure: mark worker as degraded

In [4]:
worker3_capacity_path = f"{DATA_DIR}/worker3.csv"
df = pd.read_csv(worker3_capacity_path)
df.loc[0, 'heartbeat'] = 'stale'
df.loc[0, 'cpu_pct'] = 0
df.to_csv(worker3_capacity_path, index=False)
print("Worker 3 marked as failed: heartbeat=stale, cpu_pct=0")

Worker 3 marked as failed: heartbeat=stale, cpu_pct=0


## Bring the fleet up

In [5]:
AGENT_SERVICES = ["agent-head"] + [f"agent-{n}" for n in WORKER_NODES]

def compose(*args):
    result = subprocess.run(
        ["docker", "compose", "-f", COMPOSE_FILE, "--profile", "agents", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"`docker compose {' '.join(args)}` failed with exit "
            f"{result.returncode} - see the Compose output above."
        )
    return result

if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - not launching anything.")
    print("If the wait below times out, check the Gustavo deployment.")
else:
    missing_llm = [k for k in ("LLM_BASE_URL", "LLM_MODEL")
                   if not os.environ.get(k, "").strip()]
    if missing_llm:
        raise RuntimeError(
            f"{', '.join(missing_llm)} not set - the agent head cannot start "
            f"without them.\nSet these in examples/notebooks/.env and restart "
            f"the stack (docker compose up -d), then re-run this notebook."
        )
    
    print("bringing up:", ", ".join(AGENT_SERVICES))
    #compose("up", "-d", "--no-deps", *AGENT_SERVICES)
    print("containers started - waiting for them to register below")

Gustavo owns this fleet - not launching anything.
If the wait below times out, check the Gustavo deployment.


## Wait for the fleet to come online

In [6]:
from scarlets.messaging import Messenger

FRESH_SECONDS = 90

_probe = Messenger(HEAD_BUS, agentId="notebook_probe")

def live_agents():
    now = time.time()
    return {
        agent_id
        for agent_id, record in _probe.GatherStatus().items()
        if now - record.get("ts", 0) < FRESH_SECONDS
    }

def wait_until_live(timeout):
    deadline = time.time() + timeout
    while True:
        online  = live_agents()
        missing = EXPECTED_AGENT_IDS - online
        if not missing:
            print(f"fleet online: {sorted(EXPECTED_AGENT_IDS)}")
            return online
        if time.time() >= deadline:
            where = ("the Gustavo deployment" if GUSTAVO_MANAGED_AGENTS
                     else f"docker compose -f {COMPOSE_FILE} --profile agents logs")
            raise TimeoutError(
                f"after {timeout}s these agents are still not live on {HEAD_BUS!r}: "
                f"{sorted(missing)}\nSeen on the bus: {sorted(live_agents())}\n"
                f"Check {where}."
            )
        time.sleep(2)

wait_until_live(timeout=180 if GUSTAVO_MANAGED_AGENTS else 120)

fleet online: ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']


{'__composer_api__',
 'loop-agent-head_head',
 'loop-agent-worker_worker1',
 'loop-agent-worker_worker2',
 'loop-agent-worker_worker3',
 'loop-agent-worker_worker4',
 'notebook_probe'}

## Ask the LLM head to orchestrate self-healing

The head reads capacity reports, detects failures, and redistributes work.

In [7]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


# Two corrections to the original prompt, both learned by running it.
#
# 1. It described FILE PATHS ("/data/{worker}.csv"), but query_feature
#    addresses a source by NAME, resolved from the worker's own source
#    config. The model reasonably turned the path into the name "worker1",
#    which does not exist, and every call died with "no worker holding data
#    source 'worker1' responded". The names are given explicitly below.
#
# 2. It asked the head to "mark tasks as pending again" and "update task
#    status" - writes. Every skill here is read-only: sum/median/combine
#    compute, query_feature and list_tags read. Nothing writes back to a
#    worker's data, so those steps could never have been carried out.
#
# What remains is the part that is actually self-healing reasoning: read
# each worker's real health, decide which are unfit, and compute how their
# load should be redistributed. Applying that decision would need a write
# skill, which is out of scope - the decision itself is the deliverable.
_sources = ", ".join(f"{n}_health" for n in WORKER_NODES)
answer = await ask_scarlet_agent(
    f"Use query_feature on these data sources: {_sources}. "
    "Each has columns cpu_pct, active_tasks, estimated_capacity, heartbeat. "
    # Latest row only. Asking for the whole table pulled 10 health reports
    # per worker - 40 rows - into the conversation when only the current
    # state matters. Every later turn then carries that payload, which
    # slows each call and widens the window for the endpoint's 120s proxy
    # timeout to fire.
    # 3. The SELECT list has to carry estimated_capacity even though the
    #    report does not mention it. Observed: the prompt said "report
    #    cpu_pct, active_tasks and heartbeat", the model selected exactly
    #    those three, and then could not do the final redistribution step
    #    at all - it ended by asking "would you like me to query for
    #    estimated_capacity?" instead of answering. Every column the
    #    reasoning needs must be requested in the same query, because
    #    going back for a forgotten column costs a whole extra turn.
    "Select all four of those columns, with ORDER BY timestamp DESC "
    "LIMIT 1 so you get only the most recent report from each. "
    "Report for each worker its cpu_pct, active_tasks and heartbeat. "
    "Then say which workers are unhealthy (heartbeat not 'alive', or cpu_pct "
    "at or above 85), and how the unhealthy workers' active_tasks should be "
    "divided among the healthy ones in proportion to their spare capacity."
)
print(answer)

Based on the most recent health reports from each worker:

- **Worker 1**: cpu_pct=30, active_tasks=2, heartbeat="alive" (healthy)
- **Worker 2**: cpu_pct=35, active_tasks=4, heartbeat="alive" (healthy)
- **Worker 3**: cpu_pct=96, active_tasks=8, heartbeat="alive" (unhealthy - cpu_pct ≥ 85)
- **Worker 4**: cpu_pct=39, active_tasks=2, heartbeat="alive" (healthy)

**Unhealthy Workers**: Worker 3 is unhealthy due to high CPU usage (96% ≥ 85%), even though its heartbeat is "alive".

**Spare Capacity Calculation**:
- Worker 1 spare capacity: 70 - 30 = 40
- Worker 2 spare capacity: 65 - 35 = 30
- Worker 3 spare capacity: 4 - 96 = -92 (overloaded)
- Worker 4 spare capacity: 61 - 39 = 22

Total spare capacity among healthy workers: 40 + 30 + 22 = 92

**Proportional Distribution**:
- Worker 1 gets: 40/92 ≈ 43.48% of Worker 3's tasks
- Worker 2 gets: 30/92 ≈ 32.61% of Worker 3's tasks
- Worker 4 gets: 22/92 ≈ 23.91% of Worker 3's tasks

**Task Reassignment**:
Worker 3 has 8 active tasks that nee

## Teardown

In [8]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    #compose("rm", "-s", "-f", *AGENT_SERVICES)
    print("agent containers stopped and removed")

Gustavo owns this fleet - leaving it alone.
